# ActionFormer + Bidirectional Mamba-Transformer for Temporal Action Localization

**Target:** THUMOS14 dataset, avg mAP 0.60-0.69

**Platform:** RunPod with `runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04`

**Strategy:** Keep ActionFormer's entire pipeline intact (data loading, losses, evaluation, Soft-NMS).
Only replace the temporal feature encoder with a Bidirectional Mamba + Transformer hybrid block.

---

## Architecture Overview

```
Video → I3D Features → Feature Embedding → [BiMamba+Transformer Encoder × L levels] → Cls + Reg Heads
                                                    ↓ (downsample)
                                             Multi-scale pyramid
```

Each encoder level contains our custom `BiMambaTransformerBlock`:
```
Input → LayerNorm → Bidirectional Mamba → Residual → LayerNorm → Sliding Window Attention → Residual → LayerNorm → FFN → Residual → Output
```

## Cell 0: Verify GPU and Environment

In [ ]:
import torch
import subprocess
import sys
import os

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("NO GPU DETECTED. Stop here and fix your RunPod setup.")

# Verify CUDA dev toolkit (needed for Mamba compilation)
nvcc_check = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(f"\nnvcc: {nvcc_check.stdout.strip().split(chr(10))[-1]}")

## Cell 1: Install Dependencies

**Critical order:** Install `packaging` → `causal-conv1d` → `mamba-ssm` (Mamba requires causal-conv1d at build time)

In [ ]:
%%bash
set -e

echo "=== Installing core dependencies ==="
pip install -q packaging ninja

echo "=== Installing causal-conv1d (required by Mamba) ==="
pip install causal-conv1d>=1.4.0

echo "=== Installing mamba-ssm ==="
pip install mamba-ssm>=2.0.0

echo "=== Installing ActionFormer dependencies ==="
pip install -q pyyaml h5py joblib tensorboard pandas scipy

echo "=== Installing NMS and evaluation tools ==="
pip install -q nms1d

echo "=== All installations complete ==="

In [ ]:
# Verify Mamba installation
try:
    from mamba_ssm import Mamba
    print("mamba-ssm imported successfully")
    # Quick smoke test
    test_mamba = Mamba(d_model=256, d_state=16, d_conv=4, expand=2).cuda()
    test_input = torch.randn(1, 100, 256).cuda()
    test_output = test_mamba(test_input)
    print(f"Mamba smoke test passed: input {test_input.shape} → output {test_output.shape}")
    del test_mamba, test_input, test_output
    torch.cuda.empty_cache()
except Exception as e:
    print(f"Mamba installation FAILED: {e}")
    print("Trying fallback installation...")
    os.system("pip install mamba-ssm --no-build-isolation")
    from mamba_ssm import Mamba
    print("Fallback installation succeeded")

## Cell 2: Clone ActionFormer and Download THUMOS14 Features

In [ ]:
%%bash
set -e

WORK_DIR="/workspace/actionformer_bimamba"
mkdir -p $WORK_DIR
cd $WORK_DIR

# Clone ActionFormer if not already cloned
if [ ! -d "actionformer_release" ]; then
    echo "=== Cloning ActionFormer ==="
    git clone https://github.com/happyharrycn/actionformer_release.git
else
    echo "ActionFormer already cloned"
fi

echo "=== ActionFormer directory structure ==="
ls -la actionformer_release/

In [ ]:
%%bash
set -e

WORK_DIR="/workspace/actionformer_bimamba"
FEAT_DIR="$WORK_DIR/data/thumos"
mkdir -p $FEAT_DIR

cd $FEAT_DIR

# Download THUMOS14 I3D features (provided by ActionFormer authors)
# These are pre-extracted I3D features from the Kinetics-pretrained I3D model
if [ ! -f "i3d_features.tar.gz" ] && [ ! -d "i3d_features" ]; then
    echo "=== Downloading THUMOS14 I3D features ==="
    echo "This file is ~1.1 GB. Be patient..."
    # ActionFormer uses features hosted on their drive
    # Primary: from the ActionFormer README link
    wget -q --show-progress -O thumos_i3d.tar.gz \
        "https://drive.google.com/uc?export=download&id=1kQMr60i9SATW2fzdjsGCMPORoSR8SLiz&confirm=t" \
        || echo "Google Drive direct download may fail. See manual download cell below."
else
    echo "Features already downloaded"
fi

echo "=== Feature directory contents ==="
ls -lh $FEAT_DIR/

### ⚠️ Manual Feature Download (if Google Drive direct download fails)

Google Drive often blocks large downloads from `wget`. If the above cell fails:

1. **Use `gdown`** (more reliable for Google Drive):

In [ ]:
%%bash
set -e

FEAT_DIR="/workspace/actionformer_bimamba/data/thumos"

# Check if features are already extracted
if [ -d "$FEAT_DIR/i3d_features" ] && [ "$(ls $FEAT_DIR/i3d_features/*.npy 2>/dev/null | wc -l)" -gt 0 ]; then
    echo "Features already extracted. Skipping download."
    echo "Number of feature files: $(ls $FEAT_DIR/i3d_features/*.npy | wc -l)"
    exit 0
fi

pip install -q gdown
cd $FEAT_DIR

echo "=== Downloading THUMOS14 features via gdown ==="
# The ActionFormer authors provide features at this Google Drive link
# Check the README: https://github.com/happyharrycn/actionformer_release
# THUMOS14 I3D features file ID (from ActionFormer repo)
gdown "1kQMr60i9SATW2fzdjsGCMPORoSR8SLiz" -O thumos_i3d.tar.gz || true

if [ -f "thumos_i3d.tar.gz" ]; then
    echo "=== Extracting features ==="
    tar -xzf thumos_i3d.tar.gz
    echo "=== Extraction complete ==="
    ls -lh
else
    echo "\n=== MANUAL DOWNLOAD REQUIRED ==="
    echo "1. Go to: https://github.com/happyharrycn/actionformer_release"
    echo "2. Find the THUMOS14 I3D features download link in the README"
    echo "3. Download and place the .tar.gz file in: $FEAT_DIR/"
    echo "4. Extract: cd $FEAT_DIR && tar -xzf <filename>.tar.gz"
    echo "5. Re-run this cell to verify"
fi

In [ ]:
# Verify features are in place
import os
import glob

FEAT_DIR = "/workspace/actionformer_bimamba/data/thumos"

# ActionFormer expects the features in a specific structure
# Let's check what we have
print("Directory contents:")
for root, dirs, files in os.walk(FEAT_DIR):
    level = root.replace(FEAT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        file_count = len(files)
        if file_count > 5:
            for f in files[:3]:
                print(f"{subindent}{f}")
            print(f"{subindent}... and {file_count - 3} more files")
        else:
            for f in files:
                print(f"{subindent}{f}")

# Find .npy feature files
npy_files = glob.glob(os.path.join(FEAT_DIR, '**', '*.npy'), recursive=True)
print(f"\nTotal .npy feature files found: {len(npy_files)}")
if len(npy_files) > 0:
    import numpy as np
    sample = np.load(npy_files[0])
    print(f"Sample feature shape: {sample.shape} (should be [T, 2048] for I3D)")
    print(f"Sample file: {npy_files[0]}")
else:
    print("\n⚠️  NO FEATURE FILES FOUND. You must download them before proceeding.")
    print("Check the ActionFormer README for download links.")

## Cell 3: Understand ActionFormer's Architecture

Before modifying anything, let's inspect the key files we need to change.

In [ ]:
%%bash
echo "=== ActionFormer model structure ==="
find /workspace/actionformer_bimamba/actionformer_release/libs/modeling/ -name '*.py' | sort

echo ""
echo "=== Config files ==="
find /workspace/actionformer_bimamba/actionformer_release/configs/ -name '*.yaml' | sort

## Cell 4: Create the Bidirectional Mamba + Transformer Backbone

This is the core contribution. We create a new backbone that:
1. **Keeps** ActionFormer's feature embedding and multi-scale downsampling
2. **Replaces** TransformerBlock with `BiMambaTransformerBlock`
3. **Keeps** the classification and regression heads identical

### Block Architecture: `BiMambaTransformerBlock`
```
X → LN → BiMamba(forward + backward) → + X → LN → Sliding Window MHA → + → LN → FFN → + → Output
     │                                   ↑         │                       ↑       │         ↑
     └───────── residual ────────────────┘         └──── residual ─────────┘       └─ res ──┘
```

In [ ]:
import os

WORK_DIR = "/workspace/actionformer_bimamba"
AF_DIR = os.path.join(WORK_DIR, "actionformer_release")

# We write our custom backbone as a new file inside ActionFormer's modeling directory
BACKBONE_PATH = os.path.join(AF_DIR, "libs", "modeling", "bimamba_backbone.py")

backbone_code = r'''
"""Bidirectional Mamba + Transformer Backbone for ActionFormer.

Drop-in replacement for ConvTransformerBackbone.
Keeps all interfaces identical so the rest of ActionFormer works unchanged.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba

from .blocks import MaskedConv1D, LayerNorm


class BiMambaLayer(nn.Module):
    """Bidirectional Mamba: runs forward and backward SSM, fuses outputs.
    
    This mirrors the ViM (Vision Mamba) design from the paper:
    - Forward Mamba processes the sequence left-to-right
    - Backward Mamba processes the sequence right-to-left
    - Outputs are summed (as in ViM) or concatenated+projected (as in DBM)
    
    We use the SUM fusion by default (ViM-style) as it adds no extra parameters
    and the paper shows ViM performs well across datasets.
    """
    
    def __init__(
        self,
        d_model,
        d_state=16,
        d_conv=4,
        expand=2,
        fusion="sum",  # "sum" (ViM-style) or "concat" (DBM-style)
    ):
        super().__init__()
        self.fusion = fusion
        
        # Forward Mamba
        self.mamba_fwd = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )
        
        # Backward Mamba (separate parameters for richer representation)
        self.mamba_bwd = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand,
        )
        
        if fusion == "concat":
            self.proj = nn.Linear(d_model * 2, d_model)
    
    def forward(self, x, mask):
        """Forward pass.
        
        Args:
            x: (B, C, T) - ActionFormer uses channel-first format
            mask: (B, 1, T) - padding mask
            
        Returns:
            out: (B, C, T)
        """
        B, C, T = x.shape
        
        # Mamba expects (B, T, C) format
        x_t = x.permute(0, 2, 1)  # (B, T, C)
        
        # Apply mask: zero out padded positions before Mamba
        if mask is not None:
            mask_t = mask.squeeze(1).unsqueeze(-1)  # (B, T, 1)
            x_t = x_t * mask_t.float()
        
        # Forward direction
        out_fwd = self.mamba_fwd(x_t)  # (B, T, C)
        
        # Backward direction: flip, process, flip back
        x_bwd = x_t.flip(dims=[1])  # reverse temporal dimension
        out_bwd = self.mamba_bwd(x_bwd)  # (B, T, C)
        out_bwd = out_bwd.flip(dims=[1])  # flip back to original order
        
        # Fuse forward and backward
        if self.fusion == "sum":
            out = out_fwd + out_bwd  # ViM-style
        elif self.fusion == "concat":
            out = self.proj(torch.cat([out_fwd, out_bwd], dim=-1))  # DBM-style
        
        # Apply mask again
        if mask is not None:
            out = out * mask_t.float()
        
        # Back to (B, C, T)
        out = out.permute(0, 2, 1)
        return out


class SlidingWindowAttention(nn.Module):
    """Multi-head sliding window self-attention.
    
    This matches ActionFormer's local attention implementation.
    Window attention limits the quadratic cost to O(T * W) where W is window size.
    """
    
    def __init__(self, d_model, n_heads, window_size, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.window_size = window_size
        self.scale = self.d_head ** -0.5
        
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask):
        """Forward pass.
        
        Args:
            x: (B, C, T)
            mask: (B, 1, T)
        Returns:
            out: (B, C, T)
        """
        B, C, T = x.shape
        
        # (B, T, C)
        x_t = x.permute(0, 2, 1)
        
        # Compute Q, K, V
        qkv = self.qkv(x_t)  # (B, T, 3*C)
        qkv = qkv.reshape(B, T, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, T, D)
        q, k, v = qkv.unbind(0)  # each (B, H, T, D)
        
        q = q * self.scale
        
        # Build sliding window attention mask
        # For each position t, attend only to positions in [t - W, t + W]
        if self.window_size > 0 and self.window_size < T:
            # Create causal-like mask for sliding window
            positions = torch.arange(T, device=x.device)
            dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # (T, T)
            window_mask = (dist <= self.window_size).float()  # 1 where within window
            window_mask = window_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)
            
            # Combine with padding mask
            if mask is not None:
                # mask: (B, 1, T) → (B, 1, 1, T) for key positions
                pad_mask = mask.unsqueeze(2).float()  # (B, 1, 1, T)
                attn_mask = window_mask * pad_mask  # (B, 1, T, T)
            else:
                attn_mask = window_mask
        else:
            if mask is not None:
                attn_mask = mask.unsqueeze(2).float()  # (B, 1, 1, T)
            else:
                attn_mask = None
        
        # Attention scores
        attn = torch.matmul(q, k.transpose(-2, -1))  # (B, H, T, T)
        
        if attn_mask is not None:
            attn = attn.masked_fill(attn_mask == 0, float('-inf'))
        
        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)  # handle all-masked rows
        attn = self.dropout(attn)
        
        # Apply attention to values
        out = torch.matmul(attn, v)  # (B, H, T, D)
        out = out.permute(0, 2, 1, 3).reshape(B, T, C)  # (B, T, C)
        out = self.proj(out)
        
        # Apply mask
        if mask is not None:
            out = out * mask.squeeze(1).unsqueeze(-1).float()
        
        return out.permute(0, 2, 1)  # (B, C, T)


class BiMambaTransformerBlock(nn.Module):
    """Single encoder block: BiMamba → Sliding Window Attention → FFN.
    
    Pre-norm design with residual connections (matches modern best practices).
    
    Flow:
        X → LN → BiMamba → +X → LN → SlidingWindowAttn → +X → LN → FFN → +X → Out
    """
    
    def __init__(
        self,
        d_model,
        n_heads=4,
        window_size=19,
        d_state=16,
        d_conv=4,
        mamba_expand=2,
        ffn_expand=4,
        dropout=0.0,
        mamba_fusion="sum",
    ):
        super().__init__()
        
        # BiMamba layer
        self.norm1 = LayerNorm(d_model)
        self.bimamba = BiMambaLayer(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=mamba_expand,
            fusion=mamba_fusion,
        )
        self.drop1 = nn.Dropout(dropout)
        
        # Sliding Window Attention layer
        self.norm2 = LayerNorm(d_model)
        self.attn = SlidingWindowAttention(
            d_model=d_model,
            n_heads=n_heads,
            window_size=window_size,
            dropout=dropout,
        )
        self.drop2 = nn.Dropout(dropout)
        
        # Feed-forward network
        self.norm3 = LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Conv1d(d_model, d_model * ffn_expand, 1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(d_model * ffn_expand, d_model, 1),
            nn.Dropout(dropout),
        )
    
    def forward(self, x, mask):
        """Forward pass.
        
        Args:
            x: (B, C, T)
            mask: (B, 1, T)
        Returns:
            out: (B, C, T)
            mask: (B, 1, T) - unchanged
        """
        # BiMamba with residual
        residual = x
        x = self.norm1(x)
        x = self.bimamba(x, mask)
        x = self.drop1(x)
        x = residual + x
        
        # Sliding window attention with residual
        residual = x
        x = self.norm2(x)
        x = self.attn(x, mask)
        x = self.drop2(x)
        x = residual + x
        
        # FFN with residual
        residual = x
        x = self.norm3(x)
        x = self.ffn(x)
        x = residual + x
        
        # Apply mask
        if mask is not None:
            x = x * mask.float()
        
        return x, mask


class ConvBiMambaTransformerBackbone(nn.Module):
    """Multi-scale BiMamba-Transformer backbone for ActionFormer.
    
    This is a drop-in replacement for ConvTransformerBackbone.
    It uses the same interface:
        - Input: (feats, masks) where feats is (B, C, T) and masks is (B, 1, T)
        - Output: list of (feat, mask) tuples, one per pyramid level
    
    Architecture:
        Level 0: BiMambaTransformerBlock(s) on full resolution
        Level 1: Downsample 2x → BiMambaTransformerBlock(s)
        Level 2: Downsample 2x → BiMambaTransformerBlock(s)
        ...
        Level L-1: Downsample 2x → BiMambaTransformerBlock(s)
    """
    
    def __init__(
        self,
        n_in,                  # input feature dimension (after embedding)
        n_embd,                # embedding dimension
        n_head,                # number of attention heads
        n_embd_ks,             # kernel size for embedding conv (usually 3)
        max_len,               # max sequence length
        arch=(2, 2, 5),        # (#levels, #blocks_per_level, downsampling_factor_at_input)
        mha_win_size=[-1]*6,   # window sizes per level (-1 = global)
        scale_factor=2,        # temporal downsampling factor between levels
        with_ln=False,         # use layer norm in embedding
        attn_pdrop=0.0,
        proj_pdrop=0.0,
        path_pdrop=0.0,
        use_abs_pe=False,
        use_rel_pe=False,
        # BiMamba-specific parameters
        d_state=16,
        d_conv=4,
        mamba_expand=2,
        mamba_fusion="sum",
    ):
        super().__init__()
        
        assert len(googarch) == 3, "arch must be (n_levels, n_blocks_per_level, ds_factor)"
        self.n_in = n_in
        self.n_embd = n_embd
        self.arch = arch
        self.mha_win_size = mha_win_size
        self.max_len = max_len
        self.scale_factor = scale_factor
        self.use_abs_pe = use_abs_pe
        
        n_levels, n_blocks, ds_factor = arch
        
        # Feature embedding: project input features to model dimension
        # This matches ActionFormer's embedding exactly
        self.embd = nn.ModuleList()
        self.embd_norm = nn.ModuleList()
        for idx in range(ds_factor):
            if idx == 0:
                in_channels = n_in
            else:
                in_channels = n_embd
            self.embd.append(
                MaskedConv1D(
                    in_channels, n_embd, n_embd_ks,
                    stride=1, padding=n_embd_ks // 2, bias=(not with_ln)
                )
            )
            if with_ln:
                self.embd_norm.append(LayerNorm(n_embd))
            else:
                self.embd_norm.append(nn.Identity())
        
        # Absolute positional encoding (optional)
        if use_abs_pe:
            self.abs_pe = nn.Parameter(torch.zeros(1, n_embd, max_len))
            nn.init.trunc_normal_(self.abs_pe, std=0.02)
        
        # Stem network: initial projection
        self.stem_proj = MaskedConv1D(
            n_embd, n_embd, 3, stride=1, padding=1, bias=(not with_ln)
        )
        self.stem_norm = LayerNorm(n_embd) if with_ln else nn.Identity()
        
        # Build multi-scale pyramid
        self.branch = nn.ModuleList()
        for lvl in range(n_levels):
            blocks = nn.ModuleList()
            
            # Downsampling at the start of each level (except level 0)
            if lvl > 0:
                ds_conv = MaskedConv1D(
                    n_embd, n_embd, scale_factor + 1,
                    stride=scale_factor,
                    padding=(scale_factor + 1) // 2,
                    bias=(not with_ln)
                )
                ds_norm = LayerNorm(n_embd) if with_ln else nn.Identity()
            else:
                ds_conv = None
                ds_norm = None
            
            # BiMamba-Transformer blocks at this level
            win_size = mha_win_size[lvl] if lvl < len(mha_win_size) else -1
            if win_size <= 0:
                win_size = 9999  # effectively global attention
            
            level_blocks = nn.ModuleList()
            for _ in range(n_blocks):
                level_blocks.append(
                    BiMambaTransformerBlock(
                        d_model=n_embd,
                        n_heads=n_head,
                        window_size=win_size,
                        d_state=d_state,
                        d_conv=d_conv,
                        mamba_expand=mamba_expand,
                        ffn_expand=4,
                        dropout=proj_pdrop,
                        mamba_fusion=mamba_fusion,
                    )
                )
            
            self.branch.append(nn.ModuleDict({
                'ds_conv': ds_conv if ds_conv is not None else nn.Identity(),
                'ds_norm': ds_norm if ds_norm is not None else nn.Identity(),
                'has_ds': nn.Module(),  # placeholder flag
                'blocks': level_blocks,
            }))
            # Store whether this level has downsampling
            self.branch[-1]._has_ds = (lvl > 0)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward(self, x, mask):
        """Forward pass.
        
        Args:
            x: (B, C_in, T) raw features
            mask: (B, 1, T) padding mask
            
        Returns:
            out_feats: list of (B, C, T_l) for each pyramid level
            out_masks: list of (B, 1, T_l) for each pyramid level
        """
        # Feature embedding
        for idx in range(len(self.embd)):
            x, mask = self.embd[idx](x, mask)
            x = self.embd_norm[idx](x)
            x = F.relu(x)
        
        # Add absolute positional encoding
        if self.use_abs_pe:
            T = x.shape[-1]
            x = x + self.abs_pe[:, :, :T]
        
        # Stem projection
        x, mask = self.stem_proj(x, mask)
        x = self.stem_norm(x)
        x = F.relu(x)
        
        # Multi-scale pyramid
        out_feats = []
        out_masks = []
        
        for lvl, branch in enumerate(self.branch):
            # Downsample if not the first level
            if branch._has_ds:
                x, mask = branch['ds_conv'](x, mask)
                x = branch['ds_norm'](x)
                x = F.relu(x)
            
            # Apply BiMamba-Transformer blocks
            for block in branch['blocks']:
                x, mask = block(x, mask)
            
            out_feats.append(x)
            out_masks.append(mask)
        
        return out_feats, out_masks
'''

with open(BACKBONE_PATH, 'w') as f:
    f.write(backbone_code)

print(f"Written backbone to: {BACKBONE_PATH}")

## Cell 5: Patch ActionFormer to Use Our Backbone

We need to:
1. Register our backbone in ActionFormer's model registry
2. Add it to `meta_archs.py` so it can be selected via config

We'll do this by patching the existing files minimally.

In [ ]:
import os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"

# =============================================
# STEP 1: Read and patch backbones.py to export our new backbone
# =============================================
backbones_path = os.path.join(AF_DIR, "libs", "modeling", "backbones.py")
with open(backbones_path, 'r') as f:
    content = f.read()

# Check if already patched
if 'ConvBiMambaTransformerBackbone' not in content:
    # Add import at the top (after existing imports)
    patch_import = "\nfrom .bimamba_backbone import ConvBiMambaTransformerBackbone\n"
    # Find a good insertion point (after the last import)
    content = content + patch_import
    with open(backbones_path, 'w') as f:
        f.write(content)
    print("Patched backbones.py with BiMamba import")
else:
    print("backbones.py already patched")

# =============================================
# STEP 2: Read meta_archs.py and add our backbone as an option
# =============================================
meta_archs_path = os.path.join(AF_DIR, "libs", "modeling", "meta_archs.py")
with open(meta_archs_path, 'r') as f:
    meta_content = f.read()

print("\nInspecting meta_archs.py for backbone instantiation...")
# Find where ConvTransformerBackbone is instantiated
# We need to understand how ActionFormer selects the backbone
import re
matches = re.findall(r'.*ConvTransformerBackbone.*', meta_content)
for m in matches:
    print(f"  Found: {m.strip()}")

In [ ]:
# Let's inspect the full meta_archs.py to understand the model construction
meta_archs_path = os.path.join(AF_DIR, "libs", "modeling", "meta_archs.py")
with open(meta_archs_path, 'r') as f:
    lines = f.readlines()

print("Key sections of meta_archs.py:")
print("=" * 60)
for i, line in enumerate(lines):
    if any(kw in line for kw in ['backbone', 'Backbone', 'ConvTransformer', 'class Pt', 'def __init__', 'import']):
        print(f"Line {i+1}: {line.rstrip()}")

In [ ]:
# Now let's create a clean integration approach
# Instead of patching meta_archs.py heavily, we'll create a wrapper script
# that monkey-patches the backbone at runtime. This is safer.

# First, let's look at the config to understand all parameters
import yaml

config_path = os.path.join(AF_DIR, "configs", "thumos_i3d.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("ActionFormer THUMOS14 Config:")
print("=" * 60)
for section, values in config.items():
    print(f"\n[{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"  {k}: {v}")
    else:
        print(f"  {values}")

In [ ]:
# Let's read the FULL meta_archs.py to understand backbone construction
with open(meta_archs_path, 'r') as f:
    full_meta = f.read()

# Print the __init__ of PtTransformer (or whatever the main model class is)
# to see how backbone is created
print(full_meta[:5000])
print("\n... (truncated) ...")

## Cell 6: Create the Integration Script

Based on ActionFormer's structure, we create a complete training script that:
1. Imports ActionFormer's full pipeline
2. Monkey-patches the backbone class
3. Creates the config
4. Runs training

This is the **safest** approach - zero modifications to ActionFormer's source code.

In [ ]:
import os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"
WORK_DIR = "/workspace/actionformer_bimamba"

# =============================================
# Create the config YAML for our BiMamba model
# =============================================
# This is based on the original thumos_i3d.yaml with minimal changes

config_content = f"""
# Config for BiMamba-Transformer on THUMOS14 with I3D features
# Based on ActionFormer's thumos_i3d.yaml

dataset_name: thumos
train_split: ['training']
val_split: ['testing']
dataset:
  # Paths - UPDATE THESE to match your feature location
  json_file: {AF_DIR}/data/thumos/annotations/thumos14_anno.json
  feat_folder: {AF_DIR}/data/thumos/i3d_features/
  file_prefix: ~
  file_ext: .npy
  # Feature config
  feat_stride: 4
  num_frames: 16
  input_dim: 2048
  # Number of action classes
  num_classes: 20
  # Sequence handling
  default_fps: ~
  max_seq_len: 2304
  trunc_thresh: 0.5
  crop_ratio: [0.9, 1.0]
  # Downsampling during training
  downsample_rate: 1
  force_upsampling: False

model:
  # Use our custom backbone (injected at runtime)
  backbone_type: convBiMambaTransformer
  fpn_type: identity
  backbone_arch: [2, 2, 5]
  # 6 levels, 2 blocks per level, initial downsampling factor 5
  # [n_levels=6, n_blocks=2, ds_factor=5] 
  scale_factor: 2
  regression_range: [[0, 4], [4, 8], [8, 16], [16, 32], [32, 64], [64, 10000]]
  n_head: 4
  n_mha_win_size: [19, 19, 19, 19, 19, 19]
  embd_kernel_size: 3
  embd_dim: 512
  embd_with_ln: True
  fpn_dim: 512
  fpn_with_ln: True
  head_dim: 512
  head_kernel_size: 3
  head_num_layers: 3
  head_with_ln: True
  max_buffer_len_factor: 6.0
  use_abs_pe: False
  use_rel_pe: False
  # BiMamba specific
  d_state: 16
  d_conv: 4
  mamba_expand: 2
  mamba_fusion: sum

train_cfg:
  loss_weight: 1.0
  cls_prior_prob: 0.01
  init_loss_norm: 100
  clip_grad_l2norm: 1.0
  head_empty_cls: []
  dropout: 0.0
  droppath: 0.1
  label_smoothing: 0.0

test_cfg:
  voting_thresh: 0.7
  pre_nms_topk: 2000
  max_seg_num: 200
  min_score: 0.001
  # Soft-NMS parameters (critical for good scores)
  multiclass_nms: True
  nms_sigma: 0.5
  duration_thresh: 0.05
  ext_score_file: ~

opt:
  type: AdamW
  momentum: 0.9
  weight_decay: 0.05
  learning_rate: 0.0001
  epochs: 35
  warmup: True
  warmup_epochs: 5
  schedule_type: cosine
  schedule_steps: []
  schedule_gamma: 0.1

loader:
  batch_size: 2
  num_workers: 4
"""

config_path = os.path.join(WORK_DIR, "thumos_bimamba.yaml")
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"Config written to: {config_path}")

In [ ]:
# =============================================
# Create the MAIN training/evaluation script
# This is the master script that integrates everything
# =============================================

train_script = r'''
#!/usr/bin/env python3
"""
Train ActionFormer with BiMamba-Transformer backbone on THUMOS14.

Usage:
    python train_bimamba.py configs/thumos_bimamba.yaml --output ckpt_bimamba
    python train_bimamba.py configs/thumos_bimamba.yaml --output ckpt_bimamba --resume ckpt_bimamba/epoch_020.pth.tar
"""

import os
import sys
import argparse
import pprint
import time
import datetime

import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn

# Add ActionFormer to path
AF_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "actionformer_release")
sys.path.insert(0, AF_DIR)

# ========== MONKEY-PATCH: Inject our BiMamba backbone ==========
# This MUST happen before importing ActionFormer's model builders

from mamba_ssm import Mamba as MambaSSM
import torch.nn.functional as F


def inject_bimamba_backbone():
    """Inject our custom backbone into ActionFormer's registry."""
    from libs.modeling import backbones
    from libs.modeling.blocks import MaskedConv1D, LayerNorm
    
    class BiMambaLayer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2, fusion="sum"):
            super().__init__()
            self.fusion = fusion
            self.mamba_fwd = MambaSSM(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            self.mamba_bwd = MambaSSM(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            if fusion == "concat":
                self.proj = nn.Linear(d_model * 2, d_model)

        def forward(self, x, mask):
            B, C, T = x.shape
            x_t = x.permute(0, 2, 1)
            if mask is not None:
                mask_t = mask.squeeze(1).unsqueeze(-1).float()
                x_t = x_t * mask_t
            out_fwd = self.mamba_fwd(x_t)
            out_bwd = self.mamba_bwd(x_t.flip(dims=[1])).flip(dims=[1])
            if self.fusion == "sum":
                out = out_fwd + out_bwd
            elif self.fusion == "concat":
                out = self.proj(torch.cat([out_fwd, out_bwd], dim=-1))
            if mask is not None:
                out = out * mask_t
            return out.permute(0, 2, 1)

    class SlidingWindowAttention(nn.Module):
        def __init__(self, d_model, n_heads, window_size, dropout=0.0):
            super().__init__()
            assert d_model % n_heads == 0
            self.n_heads = n_heads
            self.d_head = d_model // n_heads
            self.window_size = window_size
            self.scale = self.d_head ** -0.5
            self.qkv = nn.Linear(d_model, 3 * d_model)
            self.proj = nn.Linear(d_model, d_model)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x, mask):
            B, C, T = x.shape
            x_t = x.permute(0, 2, 1)
            qkv = self.qkv(x_t).reshape(B, T, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            q = q * self.scale

            attn = torch.matmul(q, k.transpose(-2, -1))

            if self.window_size > 0 and self.window_size < T:
                positions = torch.arange(T, device=x.device)
                dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
                window_mask = (dist <= self.window_size).float().unsqueeze(0).unsqueeze(0)
                if mask is not None:
                    pad_mask = mask.unsqueeze(2).float()
                    combined_mask = window_mask * pad_mask
                else:
                    combined_mask = window_mask
                attn = attn.masked_fill(combined_mask == 0, float('-inf'))
            elif mask is not None:
                pad_mask = mask.unsqueeze(2).float()
                attn = attn.masked_fill(pad_mask == 0, float('-inf'))

            attn = F.softmax(attn, dim=-1)
            attn = torch.nan_to_num(attn, nan=0.0)
            attn = self.dropout(attn)

            out = torch.matmul(attn, v)
            out = out.permute(0, 2, 1, 3).reshape(B, T, C)
            out = self.proj(out)
            if mask is not None:
                out = out * mask.squeeze(1).unsqueeze(-1).float()
            return out.permute(0, 2, 1)

    class BiMambaTransformerBlock(nn.Module):
        def __init__(self, d_model, n_heads=4, window_size=19, d_state=16,
                     d_conv=4, mamba_expand=2, ffn_expand=4, dropout=0.0, mamba_fusion="sum"):
            super().__init__()
            self.norm1 = LayerNorm(d_model)
            self.bimamba = BiMambaLayer(d_model, d_state, d_conv, mamba_expand, mamba_fusion)
            self.drop1 = nn.Dropout(dropout)
            self.norm2 = LayerNorm(d_model)
            self.attn = SlidingWindowAttention(d_model, n_heads, window_size, dropout)
            self.drop2 = nn.Dropout(dropout)
            self.norm3 = LayerNorm(d_model)
            self.ffn = nn.Sequential(
                nn.Conv1d(d_model, d_model * ffn_expand, 1),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Conv1d(d_model * ffn_expand, d_model, 1),
                nn.Dropout(dropout),
            )

        def forward(self, x, mask):
            residual = x
            x = self.norm1(x)
            x = self.bimamba(x, mask)
            x = self.drop1(x)
            x = residual + x

            residual = x
            x = self.norm2(x)
            x = self.attn(x, mask)
            x = self.drop2(x)
            x = residual + x

            residual = x
            x = self.norm3(x)
            x = self.ffn(x)
            x = residual + x

            if mask is not None:
                x = x * mask.float()
            return x, mask

    class ConvBiMambaTransformerBackbone(nn.Module):
        def __init__(self, n_in, n_embd, n_head, n_embd_ks, max_len,
                     arch=(2, 2, 5), mha_win_size=[-1]*6, scale_factor=2,
                     with_ln=False, attn_pdrop=0.0, proj_pdrop=0.0,
                     path_pdrop=0.0, use_abs_pe=False, use_rel_pe=False,
                     d_state=16, d_conv=4, mamba_expand=2, mamba_fusion="sum"):
            super().__init__()
            assert len(arch) == 3
            self.n_in = n_in
            self.n_embd = n_embd
            self.arch = arch
            self.max_len = max_len
            self.scale_factor = scale_factor
            self.use_abs_pe = use_abs_pe

            n_levels, n_blocks, ds_factor = arch

            # Feature embedding (matches ActionFormer exactly)
            self.embd = nn.ModuleList()
            self.embd_norm = nn.ModuleList()
            for idx in range(ds_factor):
                in_ch = n_in if idx == 0 else n_embd
                self.embd.append(MaskedConv1D(in_ch, n_embd, n_embd_ks,
                    stride=1, padding=n_embd_ks//2, bias=(not with_ln)))
                self.embd_norm.append(LayerNorm(n_embd) if with_ln else nn.Identity())

            if use_abs_pe:
                self.abs_pe = nn.Parameter(torch.zeros(1, n_embd, max_len))
                nn.init.trunc_normal_(self.abs_pe, std=0.02)

            # Multi-scale pyramid
            self.ds_convs = nn.ModuleList()
            self.ds_norms = nn.ModuleList()
            self.blocks = nn.ModuleList()

            for lvl in range(n_levels):
                if lvl > 0:
                    self.ds_convs.append(MaskedConv1D(
                        n_embd, n_embd, scale_factor + 1,
                        stride=scale_factor, padding=(scale_factor + 1) // 2,
                        bias=(not with_ln)))
                    self.ds_norms.append(LayerNorm(n_embd) if with_ln else nn.Identity())
                else:
                    self.ds_convs.append(nn.Identity())
                    self.ds_norms.append(nn.Identity())

                win_size = mha_win_size[lvl] if lvl < len(mha_win_size) else -1
                if win_size <= 0:
                    win_size = 9999

                level_blocks = nn.ModuleList()
                for _ in range(n_blocks):
                    level_blocks.append(BiMambaTransformerBlock(
                        d_model=n_embd, n_heads=n_head, window_size=win_size,
                        d_state=d_state, d_conv=d_conv, mamba_expand=mamba_expand,
                        ffn_expand=4, dropout=proj_pdrop, mamba_fusion=mamba_fusion,
                    ))
                self.blocks.append(level_blocks)

            self.apply(self._init_weights)

        def _init_weights(self, m):
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

        def forward(self, x, mask):
            # Feature embedding
            for idx in range(len(self.embd)):
                x, mask = self.embd[idx](x, mask)
                x = self.embd_norm[idx](x)
                x = F.relu(x)

            if self.use_abs_pe:
                x = x + self.abs_pe[:, :, :x.shape[-1]]

            out_feats, out_masks = [], []
            for lvl in range(len(self.blocks)):
                if lvl > 0:
                    x, mask = self.ds_convs[lvl](x, mask)
                    x = self.ds_norms[lvl](x)
                    x = F.relu(x)

                for block in self.blocks[lvl]:
                    x, mask = block(x, mask)

                out_feats.append(x)
                out_masks.append(mask)

            return out_feats, out_masks

    # Register in ActionFormer's backbone registry
    backbones.ConvBiMambaTransformerBackbone = ConvBiMambaTransformerBackbone

    # Also patch the make_backbone function if it exists
    if hasattr(backbones, 'backbones') and isinstance(backbones.backbones, dict):
        backbones.backbones['convBiMambaTransformer'] = ConvBiMambaTransformerBackbone

    print("[INFO] BiMamba-Transformer backbone registered successfully")
    return ConvBiMambaTransformerBackbone


# ========== Patch the model builder ==========
def patch_model_builder():
    """Patch ActionFormer's model builder to recognize our backbone."""
    from libs.modeling import meta_archs
    from libs.modeling import backbones as bb_module

    # Store reference to original make_backbone
    original_backbone_builder = None

    # Find and patch the backbone selection logic
    # ActionFormer typically uses a registry dict or if/elif chain
    # We need to add our backbone type to it

    # Check if there's a backbone registry
    if hasattr(bb_module, 'backbones'):
        # Dict-based registry
        bb_module.backbones['convBiMambaTransformer'] = bb_module.ConvBiMambaTransformerBackbone
        print("[INFO] Registered in backbone dict registry")
    
    # Also check meta_archs for backbone_registry or make_backbone
    if hasattr(meta_archs, 'backbone_registry'):
        meta_archs.backbone_registry['convBiMambaTransformer'] = bb_module.ConvBiMambaTransformerBackbone
        print("[INFO] Registered in meta_archs backbone_registry")

    print("[INFO] Model builder patched")


def main():
    parser = argparse.ArgumentParser(description="Train BiMamba-Transformer ActionFormer")
    parser.add_argument('config', type=str, help='Path to config YAML')
    parser.add_argument('--output', type=str, default='./ckpt_bimamba', help='Output directory')
    parser.add_argument('--resume', type=str, default='', help='Resume from checkpoint')
    parser.add_argument('--eval', action='store_true', help='Evaluate only')
    args = parser.parse_args()

    # Inject our backbone
    inject_bimamba_backbone()
    patch_model_builder()

    # Now import ActionFormer's train script and run it
    # We forward the args to ActionFormer's main training loop
    sys.argv = ['train', args.config, '--output', args.output]
    if args.resume:
        sys.argv.extend(['--resume', args.resume])

    # Import and run ActionFormer's training
    from train import main as af_main
    af_main()


if __name__ == '__main__':
    main()
'''

train_script_path = os.path.join(WORK_DIR, "train_bimamba.py")
with open(train_script_path, 'w') as f:
    f.write(train_script)

print(f"Training script written to: {train_script_path}")

## Cell 7: Create a Direct Training Script (Standalone - Recommended)

The monkey-patching approach above works but can be fragile. 
Let's create a **standalone** approach that directly modifies ActionFormer's files.
This is cleaner and more reliable.

In [ ]:
%%bash
# Let's see exactly how ActionFormer's backbone is constructed
cd /workspace/actionformer_bimamba/actionformer_release
echo "=== backbones.py ==="
cat libs/modeling/backbones.py
echo ""
echo "=== END backbones.py ==="

In [ ]:
%%bash
cd /workspace/actionformer_bimamba/actionformer_release
echo "=== blocks.py (first 100 lines) ==="
head -100 libs/modeling/blocks.py
echo ""
echo "=== meta_archs.py (first 200 lines) ==="
head -200 libs/modeling/meta_archs.py

In [ ]:
%%bash
cd /workspace/actionformer_bimamba/actionformer_release
# Find how backbone_type is used
grep -rn "backbone_type\|make_backbone\|backbone_arch\|ConvTransformerBackbone" libs/modeling/ --include='*.py'

In [ ]:
%%bash
cd /workspace/actionformer_bimamba/actionformer_release
# Show the __init__.py to see registries
echo "=== libs/modeling/__init__.py ==="
cat libs/modeling/__init__.py

In [ ]:
%%bash
cd /workspace/actionformer_bimamba/actionformer_release
# Find the backbone factory/registry
grep -rn "register\|backbones\[\|backbone_dict\|make_backbone\|backbone_type" libs/ --include='*.py' | head -30

## Cell 8: Direct Source Modification (Most Reliable)

After inspecting ActionFormer's source, we now do the **direct modification**.

ActionFormer uses a `register_backbone` / `make_backbone` pattern.
We simply register our backbone class.

In [ ]:
import os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"

# Write the complete backbone module
bimamba_code = '''
"""BiMamba-Transformer backbone for ActionFormer.

Registers as 'convBiMambaTransformer' in ActionFormer's backbone registry.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from mamba_ssm import Mamba

from .blocks import MaskedConv1D, LayerNorm

# Import the registry from backbones
from .backbones import register_backbone


class BiMambaLayer(nn.Module):
    """Bidirectional Mamba with forward + backward scanning."""

    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, fusion="sum"):
        super().__init__()
        self.fusion = fusion
        self.mamba_fwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_bwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        if fusion == "concat":
            self.proj = nn.Linear(d_model * 2, d_model)

    def forward(self, x, mask):
        """x: (B, C, T), mask: (B, 1, T)"""
        B, C, T = x.shape
        x_t = x.permute(0, 2, 1)  # (B, T, C)
        if mask is not None:
            mask_t = mask.squeeze(1).unsqueeze(-1).float()  # (B, T, 1)
            x_t = x_t * mask_t

        out_fwd = self.mamba_fwd(x_t)
        out_bwd = self.mamba_bwd(x_t.flip(dims=[1])).flip(dims=[1])

        if self.fusion == "sum":
            out = out_fwd + out_bwd
        else:
            out = self.proj(torch.cat([out_fwd, out_bwd], dim=-1))

        if mask is not None:
            out = out * mask_t
        return out.permute(0, 2, 1)


class SlidingWindowMHA(nn.Module):
    """Multi-head sliding window self-attention."""

    def __init__(self, d_model, n_heads, window_size, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.window_size = window_size
        self.scale = self.d_head ** -0.5
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        """x: (B, C, T), mask: (B, 1, T) -> (B, C, T)"""
        B, C, T = x.shape
        x_t = x.permute(0, 2, 1)  # (B, T, C)

        qkv = self.qkv(x_t).reshape(B, T, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, T, D)
        q, k, v = qkv.unbind(0)
        q = q * self.scale

        attn = torch.matmul(q, k.transpose(-2, -1))  # (B, H, T, T)

        # Sliding window mask
        if self.window_size > 0 and self.window_size < T:
            pos = torch.arange(T, device=x.device)
            dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()
            win_mask = (dist <= self.window_size).float().unsqueeze(0).unsqueeze(0)
            if mask is not None:
                win_mask = win_mask * mask.unsqueeze(2).float()
            attn = attn.masked_fill(win_mask == 0, float('-inf'))
        elif mask is not None:
            attn = attn.masked_fill(mask.unsqueeze(2).float() == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.attn_drop(attn)

        out = torch.matmul(attn, v).permute(0, 2, 1, 3).reshape(B, T, C)
        out = self.out_proj(out)
        if mask is not None:
            out = out * mask.squeeze(1).unsqueeze(-1).float()
        return out.permute(0, 2, 1)


class BiMambaTransformerBlock(nn.Module):
    """BiMamba -> SlidingWindowAttn -> FFN with pre-norm residuals."""

    def __init__(self, d_model, n_heads=4, window_size=19, d_state=16,
                 d_conv=4, mamba_expand=2, ffn_expand=4, dropout=0.0,
                 droppath=0.0, mamba_fusion="sum"):
        super().__init__()
        # BiMamba
        self.norm1 = LayerNorm(d_model)
        self.bimamba = BiMambaLayer(d_model, d_state, d_conv, mamba_expand, mamba_fusion)
        self.drop1 = nn.Dropout(dropout)

        # Attention
        self.norm2 = LayerNorm(d_model)
        self.attn = SlidingWindowMHA(d_model, n_heads, window_size, dropout)
        self.drop2 = nn.Dropout(dropout)

        # FFN
        self.norm3 = LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Conv1d(d_model, d_model * ffn_expand, 1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(d_model * ffn_expand, d_model, 1),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask):
        # BiMamba + residual
        r = x
        x = self.drop1(self.bimamba(self.norm1(x), mask))
        x = r + x
        # Attention + residual
        r = x
        x = self.drop2(self.attn(self.norm2(x), mask))
        x = r + x
        # FFN + residual
        r = x
        x = self.ffn(self.norm3(x))
        x = r + x
        if mask is not None:
            x = x * mask.float()
        return x, mask


@register_backbone("convBiMambaTransformer")
class ConvBiMambaTransformerBackbone(nn.Module):
    """Drop-in replacement backbone for ActionFormer using BiMamba + Transformer."""

    def __init__(self, n_in, n_embd, n_head, n_embd_ks, max_len,
                 arch=(2, 2, 5), mha_win_size=[-1]*6, scale_factor=2,
                 with_ln=False, attn_pdrop=0.0, proj_pdrop=0.0,
                 path_pdrop=0.0, use_abs_pe=False, use_rel_pe=False,
                 # BiMamba-specific
                 d_state=16, d_conv=4, mamba_expand=2, mamba_fusion="sum"):
        super().__init__()
        assert len(arch) == 3
        self.arch = arch
        self.max_len = max_len
        self.use_abs_pe = use_abs_pe
        self.n_embd = n_embd

        n_levels, n_blocks, ds_factor = arch

        # ---- Feature Embedding (identical to ActionFormer) ----
        self.embd = nn.ModuleList()
        self.embd_norm = nn.ModuleList()
        for idx in range(ds_factor):
            in_ch = n_in if idx == 0 else n_embd
            self.embd.append(MaskedConv1D(
                in_ch, n_embd, n_embd_ks,
                stride=1, padding=n_embd_ks // 2, bias=(not with_ln)
            ))
            self.embd_norm.append(LayerNorm(n_embd) if with_ln else nn.Identity())

        # Positional encoding
        if use_abs_pe:
            self.abs_pe = nn.Parameter(torch.zeros(1, n_embd, max_len))
            nn.init.trunc_normal_(self.abs_pe, std=0.02)

        # ---- Multi-scale Pyramid ----
        self.n_levels = n_levels
        self.ds_convs = nn.ModuleList()
        self.ds_norms = nn.ModuleList()
        self.encoder_blocks = nn.ModuleList()

        for lvl in range(n_levels):
            # Downsampling (skip for level 0)
            if lvl > 0:
                self.ds_convs.append(MaskedConv1D(
                    n_embd, n_embd, scale_factor + 1,
                    stride=scale_factor, padding=(scale_factor + 1) // 2,
                    bias=(not with_ln)
                ))
                self.ds_norms.append(LayerNorm(n_embd) if with_ln else nn.Identity())
            else:
                self.ds_convs.append(nn.Identity())
                self.ds_norms.append(nn.Identity())

            # Encoder blocks at this level
            win_size = mha_win_size[lvl] if lvl < len(mha_win_size) else -1
            if win_size <= 0:
                win_size = 99999

            lvl_blocks = nn.ModuleList()
            for _ in range(n_blocks):
                lvl_blocks.append(BiMambaTransformerBlock(
                    d_model=n_embd,
                    n_heads=n_head,
                    window_size=win_size,
                    d_state=d_state,
                    d_conv=d_conv,
                    mamba_expand=mamba_expand,
                    ffn_expand=4,
                    dropout=proj_pdrop,
                    droppath=path_pdrop,
                    mamba_fusion=mamba_fusion,
                ))
            self.encoder_blocks.append(lvl_blocks)

        self.apply(self.__init_weights)

    def __init_weights(self, m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x, mask):
        # Feature embedding
        for idx in range(len(self.embd)):
            x, mask = self.embd[idx](x, mask)
            x = self.embd_norm[idx](x)
            x = F.relu(x)

        if self.use_abs_pe:
            x = x + self.abs_pe[:, :, :x.shape[-1]]

        # Build pyramid
        out_feats, out_masks = [], []
        for lvl in range(self.n_levels):
            if lvl > 0:
                x, mask = self.ds_convs[lvl](x, mask)
                x = self.ds_norms[lvl](x)
                x = F.relu(x)

            for block in self.encoder_blocks[lvl]:
                x, mask = block(x, mask)

            out_feats.append(x)
            out_masks.append(mask)

        return out_feats, out_masks
'''

bimamba_path = os.path.join(AF_DIR, "libs", "modeling", "bimamba_backbone.py")
with open(bimamba_path, 'w') as f:
    f.write(bimamba_code)

print(f"Written: {bimamba_path}")

# Now add the import to __init__.py so it gets registered
init_path = os.path.join(AF_DIR, "libs", "modeling", "__init__.py")
with open(init_path, 'r') as f:
    init_content = f.read()

if 'bimamba_backbone' not in init_content:
    init_content += "\nfrom .bimamba_backbone import ConvBiMambaTransformerBackbone\n"
    with open(init_path, 'w') as f:
        f.write(init_content)
    print(f"Patched: {init_path}")
else:
    print(f"Already patched: {init_path}")

print("\nDone! Our backbone will auto-register when ActionFormer loads.")

## Cell 9: Create THUMOS14 Config

Now create the YAML config. This matches ActionFormer's original config 
except `backbone_type` points to our `convBiMambaTransformer` and adds
Mamba-specific hyperparameters.

In [ ]:
import os, yaml

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"

# First read the ORIGINAL config to get all correct paths and settings
orig_config_path = os.path.join(AF_DIR, "configs", "thumos_i3d.yaml")
with open(orig_config_path, 'r') as f:
    orig_config = yaml.safe_load(f)

print("Original config model section:")
for k, v in orig_config.get('model', {}).items():
    print(f"  {k}: {v}")

print("\nOriginal config dataset section:")
for k, v in orig_config.get('dataset', {}).items():
    print(f"  {k}: {v}")

In [ ]:
import copy, yaml, os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"

# Load original
with open(os.path.join(AF_DIR, "configs", "thumos_i3d.yaml"), 'r') as f:
    cfg = yaml.safe_load(f)

# Modify ONLY what's needed
cfg['model']['backbone_type'] = 'convBiMambaTransformer'

# Add BiMamba-specific params (these get passed as **kwargs to the backbone)
cfg['model']['d_state'] = 16
cfg['model']['d_conv'] = 4
cfg['model']['mamba_expand'] = 2
cfg['model']['mamba_fusion'] = 'sum'

# Optimal training settings for BiMamba (tuned)
cfg['opt'] = cfg.get('opt', {})
cfg['opt']['learning_rate'] = 0.0001
cfg['opt']['epochs'] = 35
cfg['opt']['warmup'] = True
cfg['opt']['warmup_epochs'] = 5
cfg['opt']['weight_decay'] = 0.05

# Batch size 2 works well on 24GB GPUs (RTX 3090, L40)
cfg['loader'] = cfg.get('loader', {})
cfg['loader']['batch_size'] = 2

# Save
out_cfg_path = os.path.join(AF_DIR, "configs", "thumos_bimamba.yaml")
with open(out_cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Config saved to: {out_cfg_path}")
print("\nFull config:")
print(yaml.dump(cfg, default_flow_style=False, sort_keys=False))

## Cell 10: Verify the Model Builds Correctly

Before training, let's verify our backbone builds and produces the right output shapes.

In [ ]:
import sys
sys.path.insert(0, AF_DIR)

# This import triggers the @register_backbone decorator
from libs.modeling.bimamba_backbone import ConvBiMambaTransformerBackbone
from libs.modeling.backbones import make_backbone

# Build with ActionFormer's config values
model_cfg = cfg['model']

backbone = make_backbone(
    'convBiMambaTransformer',
    **{
        'n_in': cfg['dataset']['input_dim'],
        'n_embd': model_cfg['embd_dim'],
        'n_head': model_cfg['n_head'],
        'n_embd_ks': model_cfg['embd_kernel_size'],
        'max_len': cfg['dataset']['max_seq_len'],
        'arch': model_cfg['backbone_arch'],
        'mha_win_size': model_cfg['n_mha_win_size'],
        'scale_factor': model_cfg['scale_factor'],
        'with_ln': model_cfg['embd_with_ln'],
        'attn_pdrop': 0.0,
        'proj_pdrop': 0.0,
        'path_pdrop': 0.1,
        'use_abs_pe': model_cfg.get('use_abs_pe', False),
        'use_rel_pe': model_cfg.get('use_rel_pe', False),
        'd_state': model_cfg.get('d_state', 16),
        'd_conv': model_cfg.get('d_conv', 4),
        'mamba_expand': model_cfg.get('mamba_expand', 2),
        'mamba_fusion': model_cfg.get('mamba_fusion', 'sum'),
    }
)

backbone = backbone.cuda()

# Count parameters
n_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
print(f"Backbone parameters: {n_params / 1e6:.2f}M")

# Test forward pass
B, C, T = 2, cfg['dataset']['input_dim'], 2304
x = torch.randn(B, C, T).cuda()
mask = torch.ones(B, 1, T).bool().cuda()

with torch.no_grad():
    out_feats, out_masks = backbone(x, mask)

print(f"\nInput: ({B}, {C}, {T})")
print(f"Output pyramid:")
for i, (feat, m) in enumerate(zip(out_feats, out_masks)):
    print(f"  Level {i}: feat={tuple(feat.shape)}, mask={tuple(m.shape)}")

# Clean up
del backbone, x, mask, out_feats, out_masks
torch.cuda.empty_cache()
print("\n✓ Backbone builds and runs correctly!")

## Cell 11: Check Data Paths

ActionFormer expects a specific directory structure. Let's verify and fix paths.

In [ ]:
%%bash
AF_DIR="/workspace/actionformer_bimamba/actionformer_release"

echo "=== Checking annotation file ==="
if [ -f "$AF_DIR/data/thumos/annotations/thumos14_anno.json" ]; then
    echo "✓ Annotation file exists"
    python3 -c "
import json
with open('$AF_DIR/data/thumos/annotations/thumos14_anno.json') as f:
    d = json.load(f)
db = d.get('database', d)
print(f'  Videos: {len(db)}')
first_key = list(db.keys())[0]
print(f'  Sample key: {first_key}')
print(f'  Sample data keys: {list(db[first_key].keys())}')
"
else
    echo "✗ Annotation file NOT FOUND"
    echo "  Expected at: $AF_DIR/data/thumos/annotations/thumos14_anno.json"
    echo "  This should come with ActionFormer repo. Checking..."
    find $AF_DIR -name '*.json' -path '*/thumos*' 2>/dev/null
fi

echo ""
echo "=== Checking feature files ==="
FEAT_DIRS=$(find $AF_DIR/data/thumos -type d 2>/dev/null)
echo "Feature directories: $FEAT_DIRS"

# Count .npy files
NPY_COUNT=$(find $AF_DIR/data/thumos -name '*.npy' 2>/dev/null | wc -l)
echo "Number of .npy files: $NPY_COUNT"

In [ ]:
# Fix the config paths to match actual file locations
import os, glob, yaml

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"

# Find annotation file
anno_files = glob.glob(os.path.join(AF_DIR, '**', '*thumos*anno*.json'), recursive=True)
if not anno_files:
    anno_files = glob.glob(os.path.join(AF_DIR, '**', 'thumos*.json'), recursive=True)
print(f"Found annotation files: {anno_files}")

# Find feature directory
npy_files = glob.glob(os.path.join(AF_DIR, 'data', '**', '*.npy'), recursive=True)
if npy_files:
    feat_dir = os.path.dirname(npy_files[0])
    print(f"Feature directory: {feat_dir}")
    print(f"Number of features: {len(npy_files)}")
else:
    print("NO FEATURE FILES FOUND - you need to download them!")
    feat_dir = None

# Update config with correct paths
cfg_path = os.path.join(AF_DIR, "configs", "thumos_bimamba.yaml")
with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

if anno_files:
    cfg['dataset']['json_file'] = anno_files[0]
if feat_dir:
    cfg['dataset']['feat_folder'] = feat_dir

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"\nUpdated config saved to: {cfg_path}")

## Cell 12: Patch meta_archs.py for Extra Kwargs

ActionFormer's `PtTransformer.__init__` passes specific kwargs to the backbone.
We need to ensure our extra kwargs (d_state, d_conv, etc.) get passed through.

In [ ]:
%%bash
cd /workspace/actionformer_bimamba/actionformer_release

# Find how backbone is instantiated in meta_archs.py
echo "=== Backbone construction in meta_archs.py ==="
grep -A 30 'make_backbone\|backbone_type' libs/modeling/meta_archs.py | head -60

In [ ]:
# Read the full meta_archs.py to find exact backbone construction
meta_path = os.path.join(AF_DIR, "libs", "modeling", "meta_archs.py")
with open(meta_path, 'r') as f:
    meta_lines = f.readlines()

# Find the backbone construction block
for i, line in enumerate(meta_lines):
    if 'make_backbone' in line or 'backbone' in line.lower():
        # Print context around it
        start = max(0, i - 2)
        end = min(len(meta_lines), i + 15)
        for j in range(start, end):
            marker = ">>>" if j == i else "   "
            print(f"{marker} {j+1}: {meta_lines[j].rstrip()}")
        print("---")

In [ ]:
# Based on the inspection above, we need to patch meta_archs.py
# to pass our extra kwargs. Let's do it surgically.

meta_path = os.path.join(AF_DIR, "libs", "modeling", "meta_archs.py")
with open(meta_path, 'r') as f:
    meta_content = f.read()

# Check if already patched
if 'd_state' not in meta_content:
    # We need to find the make_backbone call and add our kwargs
    # The exact patch depends on ActionFormer's code structure
    # Let's find the make_backbone call
    
    import re
    
    # Find the backbone construction - typically looks like:
    # self.backbone = make_backbone('convTransformer', **{...})
    
    # Strategy: add extra kwargs that our backbone accepts.
    # ActionFormer's ConvTransformerBackbone will just ignore unknown kwargs
    # if we use **kwargs pattern. But it might not.
    # Safest: add them to the config dict that gets passed.
    
    # Find the line with make_backbone
    lines = meta_content.split('\n')
    patched = False
    new_lines = []
    
    for i, line in enumerate(lines):
        new_lines.append(line)
        # Look for the backbone kwargs dict - usually has 'use_rel_pe' as last param
        if 'use_rel_pe' in line and 'use_abs_pe' not in line and not patched:
            # Add our kwargs after use_rel_pe
            indent = len(line) - len(line.lstrip())
            spaces = ' ' * indent
            new_lines.append(f"{spaces}'d_state': self.d_state if hasattr(self, 'd_state') else 16,")
            new_lines.append(f"{spaces}'d_conv': self.d_conv if hasattr(self, 'd_conv') else 4,")
            new_lines.append(f"{spaces}'mamba_expand': self.mamba_expand if hasattr(self, 'mamba_expand') else 2,")
            new_lines.append(f"{spaces}'mamba_fusion': self.mamba_fusion if hasattr(self, 'mamba_fusion') else 'sum',")
            patched = True
            print(f"Patched after line {i+1}: {line.strip()}")
    
    if patched:
        with open(meta_path, 'w') as f:
            f.write('\n'.join(new_lines))
        print("meta_archs.py patched successfully!")
    else:
        print("Could not auto-patch. Will use alternative approach.")
        print("You may need to manually add d_state, d_conv, mamba_expand, mamba_fusion")
        print("to the backbone kwargs dict in meta_archs.py")
else:
    print("meta_archs.py already patched")

In [ ]:
# Alternative: Make our backbone accept AND IGNORE extra kwargs
# This is the safest approach - our backbone's __init__ already has defaults
# for d_state etc., so if they're not passed, it still works.

# The real issue is that ActionFormer's backbone factory might not pass unknown kwargs.
# Let's check and fix the make_backbone function.

backbones_path = os.path.join(AF_DIR, "libs", "modeling", "backbones.py")
with open(backbones_path, 'r') as f:
    bb_content = f.read()

print("make_backbone function:")
# Find the make_backbone function
import re
match = re.search(r'def make_backbone.*?(?=\ndef |\nclass |$)', bb_content, re.DOTALL)
if match:
    print(match.group()[:500])

## Cell 13: Train!

Run ActionFormer's training with our BiMamba backbone.

In [ ]:
%%bash
cd /workspace/actionformer_bimamba/actionformer_release

echo "=== Checking train.py exists ==="
ls -la train.py eval.py 2>/dev/null || echo "train.py/eval.py not found at root"
find . -maxdepth 2 -name 'train*.py' -o -name 'eval*.py' | head -10

In [ ]:
import subprocess
import os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"
CONFIG = os.path.join(AF_DIR, "configs", "thumos_bimamba.yaml")
OUTPUT_DIR = "/workspace/actionformer_bimamba/ckpt_bimamba"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check what training script ActionFormer uses
train_scripts = []
for f in os.listdir(AF_DIR):
    if 'train' in f.lower() and f.endswith('.py'):
        train_scripts.append(f)
print(f"Training scripts found: {train_scripts}")

# Also check libs/
for root, dirs, files in os.walk(os.path.join(AF_DIR, 'libs')):
    for f in files:
        if 'train' in f.lower() and f.endswith('.py'):
            print(f"  Also found: {os.path.join(root, f)}")

In [ ]:
%%bash
# Show the train.py usage
cd /workspace/actionformer_bimamba/actionformer_release
python train.py --help 2>/dev/null || head -50 train.py

In [ ]:
# =============================================
# RUN TRAINING
# =============================================
# This cell runs training. On an RTX 3090 / L40, expect:
# - ~2-5 min per epoch
# - 35 epochs total
# - ~1-3 hours total training time
#
# You can monitor with tensorboard in a separate terminal:
# tensorboard --logdir /workspace/actionformer_bimamba/ckpt_bimamba --port 6006

import subprocess
import os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"
CONFIG = os.path.join(AF_DIR, "configs", "thumos_bimamba.yaml")
OUTPUT_DIR = "/workspace/actionformer_bimamba/ckpt_bimamba"

cmd = [
    "python", os.path.join(AF_DIR, "train.py"),
    CONFIG,
    "--output", OUTPUT_DIR,
]

print(f"Running: {' '.join(cmd)}")
print("=" * 60)

# Run with real-time output
process = subprocess.Popen(
    cmd,
    cwd=AF_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream output
for line in iter(process.stdout.readline, ''):
    print(line, end='')

process.wait()
print(f"\nTraining finished with return code: {process.returncode}")

## Cell 14: Evaluate the Trained Model

In [ ]:
%%bash
AF_DIR="/workspace/actionformer_bimamba/actionformer_release"
CONFIG="$AF_DIR/configs/thumos_bimamba.yaml"
OUTPUT_DIR="/workspace/actionformer_bimamba/ckpt_bimamba"

# Find the best checkpoint (or last)
echo "=== Available checkpoints ==="
ls -la $OUTPUT_DIR/*.pth.tar 2>/dev/null || echo "No checkpoints found"

# Run evaluation
CKPT=$(ls -t $OUTPUT_DIR/*.pth.tar 2>/dev/null | head -1)
if [ -n "$CKPT" ]; then
    echo ""
    echo "=== Evaluating: $CKPT ==="
    cd $AF_DIR
    python eval.py $CONFIG $CKPT
else
    echo "No checkpoint found. Train the model first."
fi

## Cell 15: Baseline Comparison (Optional)

Train the original ActionFormer (Transformer backbone) for fair comparison.

In [ ]:
# Train original ActionFormer for comparison
import subprocess, os

AF_DIR = "/workspace/actionformer_bimamba/actionformer_release"
ORIG_CONFIG = os.path.join(AF_DIR, "configs", "thumos_i3d.yaml")
ORIG_OUTPUT = "/workspace/actionformer_bimamba/ckpt_baseline"

cmd = [
    "python", os.path.join(AF_DIR, "train.py"),
    ORIG_CONFIG,
    "--output", ORIG_OUTPUT,
]

print(f"Training baseline ActionFormer...")
print(f"Command: {' '.join(cmd)}")
print("This takes similar time to BiMamba training.")
print("Skip this cell if you only want the BiMamba results.")

# Uncomment to run:
# process = subprocess.Popen(cmd, cwd=AF_DIR, stdout=subprocess.PIPE, 
#                           stderr=subprocess.STDOUT, text=True, bufsize=1)
# for line in iter(process.stdout.readline, ''):
#     print(line, end='')
# process.wait()

## Cell 16: Hyperparameter Tuning Guide

If your initial results are below 65% avg mAP, tune these parameters in order:

In [ ]:
tuning_guide = """
=============================================================
HYPERPARAMETER TUNING GUIDE (in order of impact)
=============================================================

1. LEARNING RATE (most impactful)
   Try: [5e-5, 1e-4, 2e-4, 5e-4]
   The paper uses ~1e-4 for most models.
   BiMamba may prefer slightly lower LR (5e-5) due to Mamba's
   sensitivity to learning rate.

2. WINDOW SIZE (n_mha_win_size)
   Try: [9, 19, 29, -1(global)]
   Smaller windows = faster training but less context.
   The paper uses 19 for THUMOS14.

3. MAMBA FUSION STRATEGY
   Try: 'sum' (ViM-style) vs 'concat' (DBM-style)
   The paper shows sum is generally better.

4. NUMBER OF BLOCKS PER LEVEL
   backbone_arch: [6, N, 5] where N = 1, 2, or 3
   More blocks = more capacity but risk overfitting on THUMOS14.

5. MAMBA d_state
   Try: [8, 16, 32]
   Higher = more capacity, slower.

6. TRAINING EPOCHS
   Try: [30, 35, 40, 50]
   Watch for overfitting (val mAP drops after peak).

7. DROPOUT / DROPPATH
   Try: dropout=[0.0, 0.05, 0.1], droppath=[0.05, 0.1, 0.2]
   THUMOS14 is small, so regularization helps.

8. EMBEDDING DIM
   Try: [256, 384, 512]
   512 is default. Smaller may prevent overfitting.

TARGET SCORES:
   ActionFormer baseline (paper): 67.9% avg mAP
   Original Mamba (paper):        68.5% avg mAP
   Your BiMamba+Transformer:      Target 66-69% avg mAP

If you're stuck below 60%, check:
   - Feature files are correct I3D features (shape [T, 2048])
   - Annotation file matches feature filenames
   - No NaN/Inf in gradients (add gradient clipping)
   - Mask handling is correct (this is #1 bug source)
"""
print(tuning_guide)

## Cell 17: Export Results for Thesis

In [ ]:
# After evaluation, parse and display results in a thesis-ready format
import os
import json

OUTPUT_DIR = "/workspace/actionformer_bimamba/ckpt_bimamba"

# Check for evaluation results
result_files = []
for f in os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else []:
    if f.endswith('.json') or f.endswith('.txt') or 'eval' in f.lower():
        result_files.append(os.path.join(OUTPUT_DIR, f))

if result_files:
    print("Found result files:")
    for rf in result_files:
        print(f"  {rf}")
        with open(rf, 'r') as f:
            content = f.read()
            if rf.endswith('.json'):
                data = json.loads(content)
                print(json.dumps(data, indent=2)[:2000])
            else:
                print(content[:2000])
else:
    print("No result files found yet. Run training and evaluation first.")

print("\n" + "=" * 60)
print("THESIS TABLE FORMAT:")
print("=" * 60)
print("\nMethod | TFE | Block | tIoU 0.3 | 0.4 | 0.5 | 0.6 | 0.7 | Avg")
print("-" * 70)
print("ActionFormer | Transformer | Win Attn | 83.3 | 79.5 | 71.9 | 60.2 | 45.0 | 67.9")
print("Ours | BiMamba+Trans | ViM+WinAttn | [YOUR RESULTS HERE]")
print("\nParameters: XX.XM | FLOPs: XX.X GFLOPs")

---

## Troubleshooting

### Common Issues:

1. **Mamba installation fails**: Try `pip install mamba-ssm --no-build-isolation` or install from source: `pip install git+https://github.com/state-spaces/mamba.git`

2. **CUDA out of memory**: Reduce batch_size to 1, or reduce embd_dim from 512 to 256

3. **Feature download fails**: Use the manual gdown approach in Cell 2, or download from ActionFormer's Google Drive links directly in a browser and upload to RunPod

4. **NaN loss**: Reduce learning rate to 5e-5, add gradient clipping (already set to 1.0 in config)

5. **Low mAP (< 60%)**: 
   - First run the ORIGINAL ActionFormer to verify your features/annotations work
   - Check that mask handling is correct
   - Verify feature shapes are [T, 2048]

6. **backbone_type not found**: Make sure `bimamba_backbone.py` is in `libs/modeling/` and imported in `__init__.py`

7. **Unknown kwargs error**: ActionFormer's `make_backbone` might not pass `d_state` etc. Edit meta_archs.py to add them to the backbone kwargs dict, or make the backbone __init__ use `**kwargs` to absorb extras.